# 12-phase13-hybrid-transformer

**neuron Phase 13** — HybridGraphLinear 를 **실제 Transformer 의 FFN 위치** 에 통합 + **RMSNorm** 도입.
Phase 8~12 는 모두 MLP-LM baseline 이었고, Phase 13 부터 standard pre-norm Transformer block 위에서 graph hidden layer 의 paradigm 이 동작하는지 검증.

핵심 가설:
1. **function preservation (Transformer 위)** — `hybrid_full_full` ≈ `plain` (RMSNorm + 표준 FFN)?
2. **dual routing 우위 재현** — Phase 12 에서 hybrid_full_around_one 이 최저 final_loss 였음. Transformer 에서도 유지?
3. **full scale-corrected (around_one × around_one)** — outer/inner 둘 다 학습 활성화 init 의 효과?
4. **RMSNorm 안정성** — LayerNorm 대비 학습 동등 또는 우위 (모든 arch 에서 NaN 없음)?

설계: 4 arch × 2 seed = 8 run, max_steps=1500.

데이터: TinyShakespeare (char-LM, block_size=64)
시드: [42, 123]
작성일: 2026-05-26
연관: Issue [#67](https://github.com/EinSofINTEREST/GraphLM/issues/67) / Phase 12 baseline PR [#66](https://github.com/EinSofINTEREST/GraphLM/pull/66)

Phase 13 의 ``identity`` outer 미지원 → 4 arch 는 plain / hybrid_full_full / hybrid_full_around_one / hybrid_around_one_around_one.

## 0. 환경 / 의존성

In [ ]:
from __future__ import annotations

import math
import statistics
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    load_tinyshakespeare_text,
)
from graphlm.neuron.hybrid_transformer_demo import (
    HybridGraphTransformerLM,
    count_parameters,
    train_hybrid_transformer_lm,
)


def safe_perplexity(loss: float, cap: float = 20.0) -> float:
    """exp(loss) overflow 방지 (학습 초반 큰 loss 또는 발산 시).

    rationale: gemini #3303153101 — `math.exp(loss)` 는 loss 큰 시점에 OverflowError.
    cap=20 → max perplexity ≈ 4.85e8 (충분히 큰 ceiling).
    """
    return math.exp(min(loss, cap))


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")

## 1. Config + 데이터

In [ ]:
# data
text = load_tinyshakespeare_text()
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
vocab_size = tokenizer.vocab_size
print(f"vocab_size = {vocab_size}, dataset size = {len(dataset)}")

# model + train hyperparameters
HIDDEN_DIM = 128
N_HEADS = 4
FFN_DIM = 256  # 2x hidden (작게 — sweep 속도 위해)
N_LAYERS = 4
GROUP_SIZE = 16  # hidden_dim/group_size = 8, ffn_dim/group_size = 16
BLOCK_SIZE = 64
BATCH_SIZE = 32
LR = 3e-4
MAX_STEPS = 1500
SEEDS = [42, 123]
ARCHS = [
    "plain",
    "hybrid_full_full",
    "hybrid_full_around_one",
    "hybrid_around_one_around_one",
]

# 모델 파라미터 수 비교 (arch 별)
print("\n== Parameter count by arch ==")
for arch in ARCHS:
    m = HybridGraphTransformerLM(
        vocab_size=vocab_size,
        hidden_dim=HIDDEN_DIM,
        n_heads=N_HEADS,
        ffn_dim=FFN_DIM,
        n_layers=N_LAYERS,
        max_seq_len=BLOCK_SIZE,
        arch=arch,
        group_size=GROUP_SIZE,
    )
    print(f"  {arch:32s} params = {count_parameters(m):,}")

## 2. Sweep 실행 (4 arch × 2 seed = 8 run)

In [ ]:
results = {}
for arch in ARCHS:
    for seed in SEEDS:
        key = (arch, seed)
        print(f"\n== arch={arch} seed={seed} ==")
        out = train_hybrid_transformer_lm(
            dataset=dataset,
            vocab_size=vocab_size,
            seed=seed,
            arch=arch,
            hidden_dim=HIDDEN_DIM,
            n_heads=N_HEADS,
            ffn_dim=FFN_DIM,
            n_layers=N_LAYERS,
            group_size=GROUP_SIZE,
            block_size=BLOCK_SIZE,
            batch_size=BATCH_SIZE,
            lr=LR,
            max_steps=MAX_STEPS,
            device=device,
        )
        results[key] = out
        print(
            f"  final_loss = {out['final_loss']:.4f}  (perplexity = {safe_perplexity(out['final_loss']):.2f})"
        )

## 3. 결과 표 + 자동 verdict

In [ ]:
print(f"{'arch':32s} {'seed':>6s} {'final_loss':>12s} {'perplexity':>12s}")
print("-" * 70)
for (arch, seed), out in results.items():
    fl = out["final_loss"]
    print(f"{arch:32s} {seed:>6d} {fl:>12.4f} {safe_perplexity(fl):>12.2f}")

# arch 별 평균 / 표준편차
print("\n== Arch-level summary (mean ± σ across seeds) ==")
summary = {}
for arch in ARCHS:
    vals = [results[(arch, s)]["final_loss"] for s in SEEDS]
    summary[arch] = (statistics.mean(vals), statistics.stdev(vals) if len(vals) > 1 else 0.0)
    m, s = summary[arch]
    print(f"  {arch:32s} {m:.4f} ± {s:.4f}  (perplexity ≈ {safe_perplexity(m):.2f})")

# 자동 verdict
print("\n== Verdict ==")
plain_loss = summary["plain"][0]
ff_loss = summary["hybrid_full_full"][0]
fa_loss = summary["hybrid_full_around_one"][0]
aa_loss = summary["hybrid_around_one_around_one"][0]

# 1. function preservation (학습 후 hybrid_full_full ≈ plain — 학습 dynamics 가 동일 sweet spot 유지)
diff_ff = abs(ff_loss - plain_loss)
verdict_1 = "PASS" if diff_ff < 0.15 else "FAIL"
print(f"1. function preservation: |hybrid_full_full - plain| = {diff_ff:.4f}  [{verdict_1}]")

# 2. scale-corrected 우위 (hybrid_full_around_one 이 plain 보다 우수 또는 동등)
verdict_2 = "PASS" if fa_loss <= plain_loss + 0.05 else "FAIL"
diff_fa = fa_loss - plain_loss
print(f"2. inner around_one ≤ plain + 0.05: diff = {diff_fa:+.4f}  [{verdict_2}]")

# 3. full scale-corrected (around_one × around_one) 안정성 (NaN 없음 + plain 근방)
verdict_3 = "PASS" if (not math.isnan(aa_loss) and aa_loss < plain_loss + 0.5) else "FAIL"
diff_aa = aa_loss - plain_loss
print(f"3. full around_one stable: diff = {diff_aa:+.4f}  [{verdict_3}]")

# 4. 모두 NaN 없이 학습 종료 (RMSNorm 안정성)
all_finite = all(not math.isnan(out["final_loss"]) for out in results.values())
verdict_4 = "PASS" if all_finite else "FAIL"
print(f"4. RMSNorm stability (all finite): {all_finite}  [{verdict_4}]")

## 4. Loss curve 시각화

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
colors = {
    "plain": "tab:gray",
    "hybrid_full_full": "tab:blue",
    "hybrid_full_around_one": "tab:orange",
    "hybrid_around_one_around_one": "tab:green",
}
window = 50  # rolling mean for smoothing

for arch in ARCHS:
    losses_per_seed = [results[(arch, s)]["losses"] for s in SEEDS]
    # rolling mean
    smoothed = []
    for losses in losses_per_seed:
        smoothed.append(
            [
                sum(losses[max(0, i - window) : i + 1]) / min(i + 1, window)
                for i in range(len(losses))
            ]
        )
    # mean ± σ across seeds
    arr = torch.tensor(smoothed)
    mean = arr.mean(dim=0)
    std = arr.std(dim=0)
    steps = list(range(len(mean)))
    color = colors[arch]
    ax.plot(steps, mean, label=arch, color=color, linewidth=1.5)
    ax.fill_between(steps, mean - std, mean + std, color=color, alpha=0.15)

ax.set_xlabel("step")
ax.set_ylabel(f"loss (rolling mean w={window})")
ax.set_title("Phase 13 — Transformer FFN: 4 arch loss curves (mean ± σ over 2 seeds)")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()

out_dir = Path("../../runs/notebook-neuron-phase13")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"saved: {out_dir / 'loss_curves.png'}")

## 5. adj_outer / adj_inner heatmap (hybrid arch 만)

각 hybrid arch 의 첫 block 의 fc1 adj_outer 와 adj_inner (block-aggregated) 학습 후 모습.

In [ ]:
hybrid_archs = [a for a in ARCHS if a != "plain"]
n_arch = len(hybrid_archs)
fig, axes = plt.subplots(n_arch, 2, figsize=(10, 3 * n_arch))
if n_arch == 1:
    axes = axes.reshape(1, -1)

for row, arch in enumerate(hybrid_archs):
    # seed 42 의 첫 block 의 fc1
    snap = results[(arch, 42)]["final_adj"][0]["fc1"]
    outer = snap["outer"].numpy()  # (G_out, G_in)
    inner = (
        snap["inner"].abs().mean(dim=(-1, -2)).numpy()
    )  # (G_out, G_in) — block-aggregated magnitude

    ax_o = axes[row, 0]
    im_o = ax_o.imshow(outer, cmap="RdBu_r", vmin=-2, vmax=2)
    ax_o.set_title(f"{arch} — adj_outer (block 0, fc1)")
    ax_o.set_xlabel("G_in")
    ax_o.set_ylabel("G_out")
    plt.colorbar(im_o, ax=ax_o, fraction=0.046)

    ax_i = axes[row, 1]
    im_i = ax_i.imshow(inner, cmap="viridis")
    ax_i.set_title(f"{arch} — |adj_inner| block-mean (block 0, fc1)")
    ax_i.set_xlabel("G_in")
    ax_i.set_ylabel("G_out")
    plt.colorbar(im_i, ax=ax_i, fraction=0.046)

plt.tight_layout()
fig.savefig(out_dir / "hybrid_adj.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"saved: {out_dir / 'hybrid_adj.png'}")

## 6. 결론 / 다음 단계

(셀 출력 보고 사용자가 채울 영역)

- function preservation: hybrid_full_full vs plain — Transformer 위에서도 동등?
- dual routing 우위: Phase 12 패턴 재현?
- adj 학습 패턴: outer / inner 가 다른 영역을 cover?

**Phase 14 후보**:
- attention 의 qkv / out 을 HybridGraphLinear 로 교체 — function-level graph 가 attention 까지 확장
- Net2Net / LiGO 식 growable transformer block — 학습 중 hidden_dim 또는 n_layers 증가